# US-04-04 — Hazelnut EXP-002 — Pooling corrigé

Export Kaggle original avec sorties conservées. Seule cette introduction a été ajoutée. La parité des embeddings est attestée sur un batch de deux normales ; la restauration CPU indépendante est consignée dans les sorties et le rapport. Ces contrôles ne mesurent pas les performances de détection.

Voir [la fiche US-04-04](../docs/epic-04/us-04-04.md). Le commit 9c93258 identifie le pipeline commun, pas le code de cet export. Aucun identifiant de version Kaggle n’est disponible ; les empreintes des exports sont enregistrées dans la provenance. Le JSON de configuration Git est descriptif : les cellules conservent leurs paramètres originaux.

Préparer une session CPU avec Internet, MVTec AD v2 et les Outputs GPU EXP-002 attachés. Le checkpoint doit être restauré par ce notebook : le chargeur US-04-05 actuel attend un autre schéma.


In [2]:

from pathlib import Path
import hashlib
import json
import torch

# ============================================
# US-04-04 / EXP-002
# Independent CPU artifact preflight
# ============================================

EXPERIMENT_ID = "EXP-002"
INPUT_ROOT = Path("/kaggle/input")

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

# Find checkpoint belonging exclusively to EXP-002
candidates = [
    path
    for path in INPUT_ROOT.rglob("patchcore_model.pt")
    if path.parent.name == EXPERIMENT_ID
    and path.parent.parent.name == "hazelnut"
]

assert len(candidates) == 1, (
    f"Expected exactly one EXP-002 checkpoint; "
    f"found {len(candidates)}: {candidates}"
)

checkpoint_path = candidates[0]
artifact_dir = checkpoint_path.parent

required_files = [
    "patchcore_model.pt",
    "hazelnut_coreset.pt",
    "coreset_report.json",
    "inference_report.json",
    "reference_prediction.npz"
]

for filename in required_files:
    path = artifact_dir / filename
    assert path.is_file(), f"Missing: {filename}"
    assert path.stat().st_size > 0

# Read original EXP-002 reports
coreset_report = json.loads(
    (artifact_dir / "coreset_report.json").read_text(
        encoding="utf-8"
    )
)

inference_report = json.loads(
    (artifact_dir / "inference_report.json").read_text(
        encoding="utf-8"
    )
)

assert coreset_report["experiment_id"] == EXPERIMENT_ID
assert inference_report["experiment_id"] == EXPERIMENT_ID

assert coreset_report["pooling_corrected"] is True
assert inference_report["pooling_corrected"] is True

assert coreset_report["input_shape"] == [320512, 1536]
assert coreset_report["memory_bank_shape"] == [3205, 1536]

# Verify source artifact integrity
checkpoint_hash = file_sha256(checkpoint_path)
reference_path_npz = artifact_dir / "reference_prediction.npz"

assert checkpoint_hash == inference_report["checkpoint_sha256"]

assert (
    file_sha256(reference_path_npz)
    == inference_report["reference_npz_sha256"]
)

assert (
    coreset_report["memory_bank_sha256"]
    == inference_report["memory_bank_sha256"]
)

# CPU-only session
assert not torch.cuda.is_available(), (
    "Disable the GPU accelerator for CPU restoration"
)

print("=== HAZELNUT EXP-002 CPU PREFLIGHT ===")
print("Experiment:", EXPERIMENT_ID)
print("Device: CPU")
print("Torch:", torch.__version__)
print("Artifact directory:", artifact_dir)
print("Checkpoint size (MiB):",
      round(checkpoint_path.stat().st_size / 1024**2, 2))
print("Checkpoint SHA256:", checkpoint_hash)
print("Memory Bank SHA256:",
      coreset_report["memory_bank_sha256"])
print("Files verified:", len(required_files))
print("Pooling corrected:",
      coreset_report["pooling_corrected"])

print("\nEXP-002 independent artifact access: PASS")


=== HAZELNUT EXP-002 CPU PREFLIGHT ===
Experiment: EXP-002
Device: CPU
Torch: 2.11.0+cpu
Artifact directory: /kaggle/input/notebooks/marwanedrissi/patchcore-hazelnut-memory-bank-exp002/patchcore/hazelnut/EXP-002
Checkpoint size (MiB): 113.86
Checkpoint SHA256: 7b4c3784434d69724e8e7fb74d587e4c8a059faa24819dd556c224d03e580a93
Memory Bank SHA256: 2b7cc437fbff34fe137be8b5c19836ae93b92615108d38f7daaa9022e63d9bec
Files verified: 5
Pooling corrected: True

EXP-002 independent artifact access: PASS


In [3]:

import sys
import json
import subprocess
from pathlib import Path
from importlib import metadata

import torch
import torchvision

# ============================================
# US-04-04 / EXP-002
# CPU dependencies validation
# ============================================

required_versions = {
    "anomalib": "2.7.0",
    "timm": "1.0.30",
    "lightning": "2.6.6",
}

def installed_version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return None

assert torch.__version__ == "2.11.0+cpu"
assert torchvision.__version__ == "0.26.0+cpu"
assert not torch.cuda.is_available()

needs_install = any(
    installed_version(name) != version
    for name, version in required_versions.items()
)

if needs_install:
    packages = [
        f"{name}=={version}"
        for name, version in required_versions.items()
    ]

    plan_path = Path(
        "/kaggle/working/exp002-install-plan.json"
    )

    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--dry-run", "--report", str(plan_path),
            *packages
        ],
        check=True
    )

    plan = json.loads(plan_path.read_text())

    planned_names = {
        item["metadata"]["name"]
        .lower().replace("_", "-")
        for item in plan["install"]
    }

    protected = {"torch", "torchvision", "triton"}

    assert not planned_names.intersection(protected), (
        "Dependency installation would alter Torch "
        "or its runtime; stop and inspect."
    )

    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            *packages
        ],
        check=True
    )

# Final verification
for name, version in required_versions.items():
    actual = installed_version(name)
    print(f"{name}: {actual}")
    assert actual == version

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.__version__ == "2.11.0+cpu"
assert torchvision.__version__ == "0.26.0+cpu"
assert not torch.cuda.is_available()

print("\nEXP-002 CPU dependencies: PASS")


anomalib: 2.7.0
timm: 1.0.30
lightning: 2.6.6
Torch: 2.11.0+cpu
Torchvision: 0.26.0+cpu
CUDA available: False

EXP-002 CPU dependencies: PASS


In [4]:

import torch
from anomalib.models.image.patchcore.torch_model import (
    PatchcoreModel
)

# ============================================
# US-04-04 / EXP-002
# Strict CPU model restoration
# ============================================

# Checkpoint integrity already verified in cell 1
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

config = checkpoint["config"]
state_dict = checkpoint["state_dict"]

# Check EXP-002 identity and corrected protocol
assert config["category"] == "hazelnut"
assert config["experiment_id"] == "EXP-002"
assert config["pooling_corrected"] is True

assert config["backbone"] == "wide_resnet50_2"
assert config["layers"] == ["layer2", "layer3"]
assert config["num_neighbors"] == 1

assert config["memory_bank_shape"] == [3205, 1536]
assert config["image_size"] == [256, 256]

assert config["memory_bank_sha256"] == (
    coreset_report["memory_bank_sha256"]
)

assert (
    config["embedding_parity_max_difference"]
    <= 1e-5
)

# Reconstruct architecture without pretrained downloads
restored_model = PatchcoreModel(
    backbone=config["backbone"],
    layers=config["layers"],
    pre_trained=False,
    num_neighbors=config["num_neighbors"]
).to("cpu")

# Strictly restore all parameters and memory bank
load_result = restored_model.load_state_dict(
    state_dict,
    strict=True
)

restored_model.feature_extractor.requires_grad_(False)
restored_model.eval()

assert not load_result.missing_keys
assert not load_result.unexpected_keys

assert tuple(restored_model.memory_bank.shape) == (
    3205, 1536
)

assert restored_model.memory_bank.device.type == "cpu"
assert torch.isfinite(restored_model.memory_bank).all()

assert not any(
    p.requires_grad
    for p in restored_model.feature_extractor.parameters()
)

assert len(restored_model.embedding_store) == 0

print("=== HAZELNUT EXP-002 CPU RESTORATION ===")
print("Experiment:", config["experiment_id"])
print("Backbone:", config["backbone"])
print("Layers:", config["layers"])
print("Pooling corrected:", config["pooling_corrected"])
print("Embedding parity:",
      config["embedding_parity_max_difference"])
print("Memory Bank:", tuple(restored_model.memory_bank.shape))
print("Device:", restored_model.memory_bank.device)
print("Missing keys:", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)

print("\nEXP-002 strict CPU restoration: PASS")


/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/anomalib/models/image/dinomaly/components/layers.py:22: FutureWarning: The anomalib.models.components.dinov2 package is deprecated and will be removed in a future release. Please use the timm-based feature extractor instead: anomalib.models.components.feature_extractors.TimmFeatureExtractor
  from anomalib.models.components.dinov2.layers import Attention, DropPath, LayerScale, MemEffAttention


=== HAZELNUT EXP-002 CPU RESTORATION ===
Experiment: EXP-002
Backbone: wide_resnet50_2
Layers: ['layer2', 'layer3']
Pooling corrected: True
Embedding parity: 0.0
Memory Bank: (3205, 1536)
Device: cpu
Missing keys: []
Unexpected keys: []

EXP-002 strict CPU restoration: PASS


In [5]:

import sys
import subprocess
import hashlib
import csv
from pathlib import Path

# ============================================
# US-04-04 / EXP-002
# CPU validation data preparation
# ============================================

REPO_DIR = Path(
    "/kaggle/working/industrial-defect-ai"
)

DATASET_ROOT = Path(
    "/kaggle/input/datasets/ipythonx/mvtec-ad"
)

SOURCE_COMMIT = config["source_commit"]
MANIFEST_SHA256 = config["manifest_sha256"]

assert SOURCE_COMMIT == (
    "9c932580cbbe27c61d92905e344c8c7e86457381"
)

assert MANIFEST_SHA256 == (
    "8f0e0d5b5bb767cd2d2e6f8a0ab10f65a959a97a38bb14bc5d9aae1d30c81fb6"
)

# 1. Clone repository if necessary
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(
        [
            "git", "clone", "--quiet",
            "https://github.com/MRdrissi/industrial-defect-ai.git",
            str(REPO_DIR)
        ],
        check=True
    )

# 2. Checkout the exact experiment source commit
subprocess.run(
    [
        "git", "-C", str(REPO_DIR),
        "checkout", "--detach", SOURCE_COMMIT
    ],
    check=True
)

actual_commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True
).strip()

assert actual_commit == SOURCE_COMMIT

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.data.mvtec_dataset import MVTecDataset
from src.data.transforms import MVTecTransform

MANIFEST = (
    REPO_DIR / "docs/data/us-02-03/hazelnut_manifest.csv"
)

assert DATASET_ROOT.is_dir()
assert MANIFEST.is_file()

# 3. Verify manifest integrity
actual_manifest_hash = hashlib.sha256(
    MANIFEST.read_bytes()
).hexdigest()

assert actual_manifest_hash == MANIFEST_SHA256

# 4. Verify reference belongs to logical Validation
with MANIFEST.open(
    encoding="utf-8-sig", newline=""
) as f:
    rows = list(csv.DictReader(f))

val_paths = {
    r["image_path"]
    for r in rows
    if r["split"] == "val"
}

train_paths = {
    r["image_path"]
    for r in rows
    if r["split"] == "train"
}

assert len(val_paths) == 78
assert len(train_paths) == 313
assert val_paths.isdisjoint(train_paths)

reference_image = inference_report["reference_image"]

assert reference_image == "hazelnut/train/good/002.png"
assert reference_image in val_paths
assert reference_image not in train_paths

# 5. Reproduce the exact preprocessing
transform = MVTecTransform(
    profile="patchcore",
    size=(256, 256)
)

val_dataset = MVTecDataset(
    dataset_root=DATASET_ROOT,
    manifest_path=MANIFEST,
    category="hazelnut",
    split="val",
    transform=transform
)

assert len(val_dataset) == 78

sample = val_dataset[0]

assert str(sample["image_path"]) == reference_image
assert int(sample["label"]) == 0
assert tuple(sample["image"].shape) == (3, 256, 256)

print("=== HAZELNUT EXP-002 CPU DATA ===")
print("Git commit:", actual_commit)
print("Manifest SHA256:", actual_manifest_hash)
print("Train normal:", len(train_paths))
print("Validation normal:", len(val_paths))
print("Reference:", sample["image_path"])
print("Image shape:", tuple(sample["image"].shape))

print("\nEXP-002 CPU data preparation: PASS")


=== HAZELNUT EXP-002 CPU DATA ===
Git commit: 9c932580cbbe27c61d92905e344c8c7e86457381
Manifest SHA256: 8f0e0d5b5bb767cd2d2e6f8a0ab10f65a959a97a38bb14bc5d9aae1d30c81fb6
Train normal: 313
Validation normal: 78
Reference: hazelnut/train/good/002.png
Image shape: (3, 256, 256)

EXP-002 CPU data preparation: PASS


HEAD is now at 9c93258 docs(EPIC-04): register completed US-04-02


In [6]:

import json
import time
import hashlib
import numpy as np
import torch
from pathlib import Path

# ============================================
# US-04-04 / EXP-002 / IDL-51
# Independent CPU / GPU comparison
# ============================================

EXPERIMENT_ID = "EXP-002"

REFERENCE_FILE = (
    artifact_dir / "reference_prediction.npz"
)

RESTORE_OUTPUT = Path(
    "/kaggle/working/patchcore/hazelnut/EXP-002"
)
RESTORE_OUTPUT.mkdir(parents=True, exist_ok=True)

SCORE_TOL = 1e-3
MAP_TOL = 1e-3
RELATIVE_MAP_TOL = 1e-4

# 1. Load EXP-002 GPU reference
with np.load(
    REFERENCE_FILE,
    allow_pickle=False
) as ref:
    gpu_score = np.asarray(
        ref["pred_score"],
        dtype=np.float32
    ).reshape(-1)

    gpu_map = np.asarray(
        ref["anomaly_map"],
        dtype=np.float32
    ).copy()

    reference_path = str(ref["image_path"].item())

assert reference_path == str(sample["image_path"])
assert gpu_score.shape == (1,)
assert gpu_map.shape == (1, 1, 256, 256)

assert np.isfinite(gpu_score).all()
assert np.isfinite(gpu_map).all()

assert np.isclose(
    float(gpu_score[0]),
    inference_report["pred_score"],
    rtol=0,
    atol=1e-6
)

# 2. Verify Memory Bank SHA256
restored_bank = (
    restored_model.memory_bank
    .detach()
    .cpu()
    .contiguous()
)

bank_hash = hashlib.sha256(
    restored_bank.numpy().tobytes()
).hexdigest()

assert bank_hash == coreset_report["memory_bank_sha256"]
assert bank_hash == config["memory_bank_sha256"]

# 3. Independent CPU inference
restored_model.eval()

input_tensor = sample["image"].unsqueeze(0).to("cpu")

start = time.perf_counter()

with torch.inference_mode():
    prediction = restored_model(input_tensor)

cpu_duration = time.perf_counter() - start

cpu_score = (
    prediction.pred_score
    .detach().cpu().numpy()
    .astype(np.float32)
    .reshape(-1)
)

cpu_map = (
    prediction.anomaly_map
    .detach().cpu().numpy()
    .astype(np.float32)
)

assert cpu_score.shape == gpu_score.shape
assert cpu_map.shape == gpu_map.shape

assert np.isfinite(cpu_score).all()
assert np.isfinite(cpu_map).all()

# 4. Numerical comparison
score_diff = float(
    np.max(np.abs(cpu_score - gpu_score))
)

map_diff = np.abs(cpu_map - gpu_map)

map_max_diff = float(map_diff.max())
map_mean_diff = float(map_diff.mean())

map_range = float(
    gpu_map.max() - gpu_map.min()
)

map_relative_diff = (
    map_max_diff / map_range
    if map_range > 0
    else map_max_diff
)

passed = (
    score_diff <= SCORE_TOL
    and map_max_diff <= MAP_TOL
    and map_relative_diff <= RELATIVE_MAP_TOL
)

# 5. Save EXP-002 restoration report
report = {
    "category": "hazelnut",
    "experiment_id": EXPERIMENT_ID,
    "protocol": "PatchCore with local pooling",
    "pooling_corrected": True,
    "device": "cpu",
    "reference_image": reference_path,
    "checkpoint_sha256": checkpoint_hash,
    "memory_bank_sha256": bank_hash,
    "gpu_score": float(gpu_score[0]),
    "cpu_score": float(cpu_score[0]),
    "score_abs_diff": score_diff,
    "map_max_abs_diff": map_max_diff,
    "map_mean_abs_diff": map_mean_diff,
    "map_relative_max_diff": map_relative_diff,
    "cpu_inference_seconds": cpu_duration,
    "tolerances": {
        "score_absolute": SCORE_TOL,
        "map_max_absolute": MAP_TOL,
        "map_relative_max": RELATIVE_MAP_TOL
    },
    "strict_load": True,
    "weights_only": True,
    "coreset_recomputed": False,
    "status": "PASS" if passed else "FAIL"
}

report_path = RESTORE_OUTPUT / "restore_report.json"

report_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

# 6. Display results
print("=== HAZELNUT EXP-002 CPU / GPU COMPARISON ===")
print("Reference:", reference_path)
print("GPU score:", float(gpu_score[0]))
print("CPU score:", float(cpu_score[0]))
print("Score absolute difference:", score_diff)
print("Map maximum difference:", map_max_diff)
print("Map mean difference:", map_mean_diff)
print("Map relative difference:", map_relative_diff)
print("CPU inference duration (s):",
      round(cpu_duration, 3))
print("Memory Bank SHA256:", bank_hash)
print("Report:", report_path)

assert passed, (
    "EXP-002 CPU/GPU numerical comparison failed"
)

print("\nINDEPENDENT EXP-002 RESTORATION: PASS")


=== HAZELNUT EXP-002 CPU / GPU COMPARISON ===
Reference: hazelnut/train/good/002.png
GPU score: 47.80107116699219
CPU score: 47.80107498168945
Score absolute difference: 3.814697265625e-06
Map maximum difference: 0.000164031982421875
Map mean difference: 3.0134106054902077e-05
Map relative difference: 5.631701028446835e-06
CPU inference duration (s): 0.664
Memory Bank SHA256: 2b7cc437fbff34fe137be8b5c19836ae93b92615108d38f7daaa9022e63d9bec
Report: /kaggle/working/patchcore/hazelnut/EXP-002/restore_report.json

INDEPENDENT EXP-002 RESTORATION: PASS


In [7]:

import json
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display

# ============================================
# US-04-04 / EXP-002
# Evidence archive export
# ============================================

gpu_dir = artifact_dir

cpu_dir = Path(
    "/kaggle/working/patchcore/hazelnut/EXP-002"
)

files = [
    (
        gpu_dir / "coreset_report.json",
        "coreset_report.json"
    ),
    (
        gpu_dir / "inference_report.json",
        "inference_report.json"
    ),
    (
        gpu_dir / "reference_prediction.npz",
        "reference_prediction.npz"
    ),
    (
        cpu_dir / "restore_report.json",
        "restore_report.json"
    )
]

# Verify CPU restoration succeeded
restore_data = json.loads(
    (cpu_dir / "restore_report.json").read_text()
)

assert restore_data["experiment_id"] == "EXP-002"
assert restore_data["pooling_corrected"] is True
assert restore_data["status"] == "PASS"

# Create archive without the large checkpoint
zip_path = Path(
    "/kaggle/working/us-04-04-exp002-evidence.zip"
)

with ZipFile(
    zip_path, "w", ZIP_DEFLATED
) as archive:
    for source, name in files:
        assert source.is_file(), (
            f"Missing artifact: {source}"
        )

        archive.write(
            source,
            arcname=name
        )

        print(f"[OK] {name}")

assert zip_path.is_file()
assert zip_path.stat().st_size > 0

print("\n=== HAZELNUT EXP-002 EVIDENCE ===")
print("Archive:", zip_path)
print("Size (KiB):",
      round(zip_path.stat().st_size / 1024, 2))

print("\nEXP-002 evidence archive: PASS")

display(FileLink(str(zip_path)))


[OK] coreset_report.json
[OK] inference_report.json
[OK] reference_prediction.npz
[OK] restore_report.json

=== HAZELNUT EXP-002 EVIDENCE ===
Archive: /kaggle/working/us-04-04-exp002-evidence.zip
Size (KiB): 223.23

EXP-002 evidence archive: PASS


/kaggle/working/us-04-04-exp002-evidence.zip